In [0]:
from pyspark.sql.functions import current_timestamp, lit
caminho = "/Volumes/workspace/default/lh_nautical"
tabela = [
    "addresses", "attributes", "brands", "categories", "customers", "employees", "fiscal_invoices", "goods_receipt_items", "goods_receipts", "locations", "order_items", "orders", "payments", "product_suppliers", "product_variants", "products", "purchase_order_items", "purchase_orders", "return_items", "returns", "stock_levels", "suppliers"
]
print("Inciando a c da camada bronze...\n")
for tabela in tabela:
    caminho_csv = f"{caminho}/{tabela}.csv"
    df_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(caminho_csv)
    #acdiona timestamp de ingestão
    df_bronze = df_raw.withColumn("_ingested_at", current_timestamp())
    #escreve no delta
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"default.bronze_{tabela}")
print(f"✅ Tabela 'bronze_{tabela}' criada com {df_bronze.count():,} registros.")

print("\n Camada Bronze finalizada com sucesso!")

Inciando a c da camada bronze...

✅ Tabela 'bronze_suppliers' criada com 25 registros.

 Camada Bronze finalizada com sucesso!


In [0]:
%sql
-- QUESTÃO 1.1: Visão geral + valores numéricos, direto de orders (sem tratamento)
SELECT
    COUNT(*) AS total_linhas,
    MIN(created_at) AS data_minima,
    MAX(created_at) AS data_maxima,
    MIN(total) AS valor_minimo,
    MAX(total) AS valor_maximo,
    AVG(total) AS valor_medio
FROM default.bronze_orders

total_linhas,data_minima,data_maxima,valor_minimo,valor_maximo,valor_medio
48998,2020-01-01T01:19:28.000Z,2026-12-31T23:43:09.000Z,32.62,127262.02,28704.992077227642


In [0]:
resultado_q1 = spark.sql("""
    SELECT
        COUNT(*) AS total_linhas,
        MIN(created_at) AS data_minima,
        MAX(created_at) AS data_maxima,
        MIN(total) AS valor_minimo,
        MAX(total) AS valor_maximo,
        AVG(total) AS valor_medio
    FROM default.bronze_orders
""")
resultado_q1.show(truncate=False)

+------------+-------------------+-------------------+------------+------------+------------------+
|total_linhas|data_minima        |data_maxima        |valor_minimo|valor_maximo|valor_medio       |
+------------+-------------------+-------------------+------------+------------+------------------+
|48998       |2020-01-01 01:19:28|2026-12-31 23:43:09|32.62       |127262.02   |28704.992077227642|
+------------+-------------------+-------------------+------------+------------+------------------+



In [0]:
import csv
import os

CAMINHO_CSVS = "/Volumes/workspace/default/lh_nautical"
CAMINHO_SAIDA = "/Volumes/workspace/default/lh_nautical/schema.sql"  

def inferir_tipo_postgres(valores_amostra):
    """Infere o tipo de coluna Postgres a partir de uma amostra de valores (strings)."""
    valores = [v for v in valores_amostra if v not in (None, "")]
    if not valores:
        return "TEXT"

    # tenta inteiro
    if all(_eh_inteiro(v) for v in valores):
        return "BIGINT"

    # tenta float
    if all(_eh_float(v) for v in valores):
        return "DOUBLE PRECISION"

    # tenta timestamp
    if all(_eh_timestamp(v) for v in valores):
        return "TIMESTAMP"

    # tenta booleano
    if all(v.lower() in ("true", "false") for v in valores):
        return "BOOLEAN"

    return "TEXT"

def _eh_inteiro(v):
    try:
        int(v)
        return True
    except ValueError:
        return False

def _eh_float(v):
    try:
        float(v)
        return True
    except ValueError:
        return False

def _eh_timestamp(v):
    formatos = ["%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d"]
    from datetime import datetime
    for fmt in formatos:
        try:
            datetime.strptime(v, fmt)
            return True
        except ValueError:
            continue
    return False

def gerar_schema_sql(caminho_csvs, caminho_saida, tamanho_amostra=200):
    arquivos_csv = [f for f in os.listdir(caminho_csvs) if f.endswith(".csv")]
    statements = []

    for arquivo in sorted(arquivos_csv):
        nome_tabela = arquivo.replace(".csv", "")
        caminho_completo = os.path.join(caminho_csvs, arquivo)

        with open(caminho_completo, newline="", encoding="utf-8") as f:
            leitor = csv.reader(f)
            cabecalho = next(leitor)
            amostras = {col: [] for col in cabecalho}

            for i, linha in enumerate(leitor):
                if i >= tamanho_amostra:
                    break
                for col, valor in zip(cabecalho, linha):
                    amostras[col].append(valor)

        colunas_sql = []
        for col in cabecalho:
            tipo = inferir_tipo_postgres(amostras[col])
            nome_col_seguro = col.strip().lower().replace(" ", "_")
            colunas_sql.append(f'    "{nome_col_seguro}" {tipo}')

        create_stmt = f'CREATE TABLE IF NOT EXISTS "{nome_tabela}" (\n' + ",\n".join(colunas_sql) + "\n);"
        statements.append(create_stmt)
        print(f"✅ Schema inferido para '{nome_tabela}' ({len(cabecalho)} colunas)")

    with open(caminho_saida, "w", encoding="utf-8") as f:
        f.write("-- Schema gerado automaticamente a partir dos CSVs (Python puro)\n")
        f.write("-- Destino: PostgreSQL\n\n")
        f.write("\n\n".join(statements))

    print(f"\n✅ Arquivo '{caminho_saida}' gerado com {len(statements)} tabelas.")

gerar_schema_sql(CAMINHO_CSVS, CAMINHO_SAIDA)

✅ Schema inferido para 'addresses' (12 colunas)
✅ Schema inferido para 'attributes' (3 colunas)
✅ Schema inferido para 'brands' (6 colunas)
✅ Schema inferido para 'categories' (7 colunas)
✅ Schema inferido para 'customers' (11 colunas)
✅ Schema inferido para 'employees' (11 colunas)
✅ Schema inferido para 'fiscal_invoices' (11 colunas)
✅ Schema inferido para 'goods_receipt_items' (4 colunas)
✅ Schema inferido para 'goods_receipts' (6 colunas)
✅ Schema inferido para 'locations' (14 colunas)
✅ Schema inferido para 'order_items' (8 colunas)
✅ Schema inferido para 'orders' (13 colunas)
✅ Schema inferido para 'payments' (9 colunas)
✅ Schema inferido para 'product_suppliers' (8 colunas)
✅ Schema inferido para 'product_variants' (12 colunas)
✅ Schema inferido para 'products' (10 colunas)
✅ Schema inferido para 'purchase_order_items' (6 colunas)
✅ Schema inferido para 'purchase_orders' (13 colunas)
✅ Schema inferido para 'return_items' (7 colunas)
✅ Schema inferido para 'returns' (10 colunas)


In [0]:
import sqlite3
import csv
import os
import shutil

CAMINHO_CSVS = "/Volumes/workspace/default/lh_nautical"
CAMINHO_DB_LOCAL = "/tmp/lh_nautical_bruto.db"          # grava local primeiro
CAMINHO_DB_VOLUME = "/Volumes/workspace/default/lh_nautical/lh_nautical_bruto.db"  # destino final

# Remove versão anterior local, se existir
if os.path.exists(CAMINHO_DB_LOCAL):
    os.remove(CAMINHO_DB_LOCAL)

conn = sqlite3.connect(CAMINHO_DB_LOCAL)
cursor = conn.cursor()

arquivos_csv = [f for f in os.listdir(CAMINHO_CSVS) if f.endswith(".csv")]

for arquivo in sorted(arquivos_csv):
    nome_tabela = arquivo.replace(".csv", "")
    caminho_completo = os.path.join(CAMINHO_CSVS, arquivo)

    with open(caminho_completo, newline="", encoding="utf-8") as f:
        leitor = csv.reader(f)
        cabecalho = next(leitor)
        linhas = list(leitor)

    colunas_sql = ", ".join([f'"{c}" TEXT' for c in cabecalho])
    cursor.execute(f'DROP TABLE IF EXISTS "{nome_tabela}"')
    cursor.execute(f'CREATE TABLE "{nome_tabela}" ({colunas_sql})')

    placeholders = ", ".join(["?"] * len(cabecalho))
    cursor.executemany(f'INSERT INTO "{nome_tabela}" VALUES ({placeholders})', linhas)

    conn.commit()
    print(f"✅ Tabela '{nome_tabela}' carregada com {len(linhas):,} registros.")

conn.close()
print("\n✅ Carregamento bruto finalizado (SQLite - local).")

# Copia o banco pronto do disco local pro Volume (só cópia de arquivo, sem I/O concorrente)
shutil.copy(CAMINHO_DB_LOCAL, CAMINHO_DB_VOLUME)
print(f"✅ Banco copiado para o Volume: {CAMINHO_DB_VOLUME}")

✅ Tabela 'addresses' carregada com 3,998 registros.
✅ Tabela 'attributes' carregada com 8 registros.
✅ Tabela 'brands' carregada com 12 registros.
✅ Tabela 'categories' carregada com 14 registros.
✅ Tabela 'customers' carregada com 2,000 registros.
✅ Tabela 'employees' carregada com 15 registros.
✅ Tabela 'fiscal_invoices' carregada com 34,365 registros.
✅ Tabela 'goods_receipt_items' carregada com 4,733 registros.
✅ Tabela 'goods_receipts' carregada com 1,548 registros.
✅ Tabela 'locations' carregada com 6 registros.
✅ Tabela 'order_items' carregada com 147,320 registros.
✅ Tabela 'orders' carregada com 48,998 registros.
✅ Tabela 'payments' carregada com 53,546 registros.
✅ Tabela 'product_suppliers' carregada com 1,520 registros.
✅ Tabela 'product_variants' carregada com 1,009 registros.
✅ Tabela 'products' carregada com 500 registros.
✅ Tabela 'purchase_order_items' carregada com 6,059 registros.
✅ Tabela 'purchase_orders' carregada com 2,000 registros.
✅ Tabela 'return_items' carre

In [0]:
conn = sqlite3.connect(CAMINHO_DB_LOCAL)  # lê do local, não do Volume
cursor = conn.cursor()

tabelas_validacao = ["customers", "orders", "order_items", "payments"]
total = 0
for t in tabelas_validacao:
    cursor.execute(f'SELECT COUNT(*) FROM "{t}"')
    n = cursor.fetchone()[0]
    total += n
    print(f"{t}: {n:,}")

print(f"\nTotal somado: {total:,}")
conn.close()

customers: 2,000
orders: 48,998
order_items: 147,320
payments: 53,546

Total somado: 251,864


In [0]:
from pyspark.sql.functions import col, to_timestamp

print("Iniciando o processamento da Camada Silver...\n")
df_orders = spark.table("default.bronze_orders") \
    .withColumn("placed_at", to_timestamp(col("placed_at"))) \
    .withColumn("created_at", to_timestamp(col("created_at"))) \
    .filter(col("status").isin("paid", "completed", "shipped"))
df_orders.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.silver_orders")
print("✅ Tabela 'silver_orders' processada (apenas pedidos válidos).")
# 2. Tratamento de Itens de Pedidos 
df_order_items = spark.table("default.bronze_order_items") \
    .withColumn("unit_price", col("unit_price").cast("double")) \
    .withColumn("quantity", col("quantity").cast("double")) \
    .withColumn("line_total", col("line_total").cast("double"))
df_order_items.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.silver_order_items")
print("✅ Tabela 'silver_order_items' processada.")
# 3. Tratamento de Devoluções C
df_returns = spark.table("default.bronze_returns") \
    .filter(col("status") == "completed")
df_returns.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.silver_returns")
print("✅ Tabela 'silver_returns' processada (apenas devoluções concluídas).")

print("\n Camada Silver finalizada")

Iniciando o processamento da Camada Silver...

✅ Tabela 'silver_orders' processada (apenas pedidos válidos).
✅ Tabela 'silver_order_items' processada.
✅ Tabela 'silver_returns' processada (apenas devoluções concluídas).

 Camada Silver finalizada


In [0]:
from pyspark.sql.functions import col, count, when, isnan

tabelas = [
    "addresses", "attributes", "brands", "categories", "customers", "employees",
    "fiscal_invoices", "goods_receipt_items", "goods_receipts", "locations",
    "order_items", "orders", "payments", "product_suppliers", "product_variants",
    "products", "purchase_order_items", "purchase_orders", "return_items",
    "returns", "stock_levels", "suppliers"
]

print("--- ANÁLISE COMPLEMENTAR: VOLUMETRIA DAS 22 TABELAS BRONZE (apoio ao dashboard) ---\n")

resumo_eda = []

for t in tabelas:
    df = spark.table(f"default.bronze_{t}")
    n_linhas = df.count()
    n_colunas = len(df.columns)
    resumo_eda.append((t, n_linhas, n_colunas))

df_resumo = spark.createDataFrame(resumo_eda, ["tabela", "n_linhas", "n_colunas"])
df_resumo.orderBy(col("n_linhas").desc()).show(22, truncate=False)

# Checagem de nulos nas tabelas centrais do negócio (as mais usadas nas questões seguintes)
tabelas_criticas = ["customers", "orders", "order_items", "products", "product_variants"]

print("--- NULOS NAS TABELAS CRÍTICAS ---\n")
for t in tabelas_criticas:
    df = spark.table(f"default.bronze_{t}")
    print(f"\nTabela: {t}")
    exprs = [
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]
    df.select(exprs).show(truncate=False)

--- ANÁLISE COMPLEMENTAR: VOLUMETRIA DAS 22 TABELAS BRONZE (apoio ao dashboard) ---

+--------------------+--------+---------+
|tabela              |n_linhas|n_colunas|
+--------------------+--------+---------+
|order_items         |147320  |9        |
|payments            |53546   |10       |
|orders              |48998   |14       |
|fiscal_invoices     |34365   |12       |
|purchase_order_items|6059    |7        |
|stock_levels        |6054    |6        |
|goods_receipt_items |4733    |5        |
|addresses           |3998    |13       |
|customers           |2000    |12       |
|purchase_orders     |2000    |14       |
|goods_receipts      |1548    |7        |
|product_suppliers   |1520    |9        |
|return_items        |1384    |8        |
|product_variants    |1009    |13       |
|returns             |980     |11       |
|products            |500     |11       |
|suppliers           |25      |13       |
|employees           |15      |12       |
|categories          |14      |8 

In [0]:
tabelas = [
    "addresses", "attributes", "brands", "categories", "customers", "employees",
    "fiscal_invoices", "goods_receipt_items", "goods_receipts", "locations",
    "order_items", "orders", "payments", "product_suppliers", "product_variants",
    "products", "purchase_order_items", "purchase_orders", "return_items",
    "returns", "stock_levels", "suppliers"
]

print("--- ANÁLISE COMPLEMENTAR: SCHEMA DAS 22 TABELAS BRONZE (apoio ao dashboard) ---\n")

for t in tabelas:
    df = spark.table(f"default.bronze_{t}")
    print(f"\n=== Tabela: bronze_{t} ===")
    df.printSchema()

--- ANÁLISE COMPLEMENTAR: SCHEMA DAS 22 TABELAS BRONZE (apoio ao dashboard) ---


=== Tabela: bronze_addresses ===
root
 |-- id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- address_type: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- street: string (nullable = true)
 |-- number: integer (nullable = true)
 |-- complement: string (nullable = true)
 |-- district: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- is_primary: boolean (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)


=== Tabela: bronze_attributes ===
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- data_type: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)


=== Tabela: bronze_brands ===
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- i

In [0]:
#Questões 4 e 5
from pyspark.sql.functions import col, sum as _sum, avg, countDistinct, expr, to_date, dayofweek, min as _min, max as _max

print("Iniciando o processamento da Camada Gold...\n")

# Aliases para facilitar os joins
o = spark.table("default.silver_orders").alias("o")
oi = spark.table("default.silver_order_items").alias("oi")
pv = spark.table("default.bronze_product_variants").alias("pv")
p = spark.table("default.bronze_products").alias("p")
c = spark.table("default.bronze_customers").alias("c")

# Base de vendas
df_vendas_completa = o \
    .join(oi, col("o.id") == col("oi.order_id")) \
    .join(pv, col("oi.product_variant_id") == col("pv.id")) \
    .join(p, col("pv.product_id") == col("p.id"))

# ============================================================
# QUESTÃO 4: Clientes Fiéis (Ticket Médio e Diversidade >= 13)
# ============================================================

# Faturamento e frequência: direto de 'orders' (sem duplicação por item)
df_faturamento = spark.table("default.silver_orders") \
    .groupBy(col("customer_id")) \
    .agg(
        _sum("total").alias("faturamento_total"),
        countDistinct("id").alias("frequencia")
    )

# Diversidade de categorias
df_diversidade = df_vendas_completa \
    .groupBy(col("o.customer_id").alias("customer_id")) \
    .agg(countDistinct("p.category_id").alias("diversidade_categorias"))

# Junta as duas bases, calcula ticket médio, filtra e desempata por customer_id
df_q4 = df_faturamento.join(df_diversidade, "customer_id") \
    .withColumn("ticket_medio", col("faturamento_total") / col("frequencia")) \
    .filter(col("diversidade_categorias") >= 13) \
    .join(c, col("customer_id") == col("c.id")) \
    .select("c.id", "c.legal_name", "ticket_medio", "diversidade_categorias", "faturamento_total", "frequencia") \
    .orderBy(col("ticket_medio").desc(), col("id").asc()) \
    .limit(10)

df_q4.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.gold_q4_clientes_fies")

print("--- QUESTÃO 4: TOP 10 CLIENTES FIÉIS (DIVERSIDADE >= 13) ---")
df_q4.show(10, truncate=False)

# TAREFA 3 (Q4): categoria que concentra mais itens comprados entre os top 10
top10_ids = [row["id"] for row in df_q4.select("id").collect()]

df_categoria_top = df_vendas_completa \
    .filter(col("o.customer_id").isin(top10_ids)) \
    .groupBy(col("p.category_id")) \
    .agg(_sum("oi.quantity").alias("total_itens")) \
    .orderBy(col("total_itens").desc())

print("--- CATEGORIA MAIS COMPRADA PELOS TOP 10 CLIENTES FIÉIS ---")
df_categoria_top.show(5, truncate=False)

# ============================================================
# QUESTÃO 5: Calendário e Dias em Português (Date Spine) — só lojas físicas
# ============================================================

# Intervalo dinâmico, a partir dos dados reais de canal 'pos'
intervalo = spark.table("default.silver_orders") \
    .filter(col("channel") == "pos") \
    .select(_min(to_date(col("placed_at"))).alias("data_min"), _max(to_date(col("placed_at"))).alias("data_max")) \
    .collect()[0]

data_min = intervalo["data_min"]
data_max = intervalo["data_max"]
print(f"\nPeríodo de análise (Q5): {data_min} até {data_max}")

df_calendario = spark.sql(f"""
    SELECT explode(sequence(to_date('{data_min}'), to_date('{data_max}'), interval 1 day)) as data_completa
""")

df_vendas_diarias = spark.table("default.silver_orders") \
    .filter(col("channel") == "pos") \
    .withColumn("data_completa", to_date(col("placed_at"))) \
    .groupBy("data_completa") \
    .agg(_sum("total").alias("venda_do_dia"))

df_q5 = df_calendario.join(df_vendas_diarias, "data_completa", "left") \
    .na.fill({"venda_do_dia": 0}) \
    .withColumn("num_dia", dayofweek(col("data_completa"))) \
    .withColumn("dia_semana_pt", expr("""
        CASE num_dia 
            WHEN 1 THEN '1. Domingo'
            WHEN 2 THEN '2. Segunda-feira'
            WHEN 3 THEN '3. Terça-feira'
            WHEN 4 THEN '4. Quarta-feira'
            WHEN 5 THEN '5. Quinta-feira'
            WHEN 6 THEN '6. Sexta-feira'
            WHEN 7 THEN '7. Sábado'
        END
    """)) \
    .groupBy("num_dia", "dia_semana_pt") \
    .agg(avg("venda_do_dia").alias("media_venda_diaria")) \
    .orderBy("num_dia")

df_q5.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.gold_q5_vendas_calendario")

print("--- QUESTÃO 5: MÉDIA DE VENDAS POR DIA DA SEMANA (LOJAS FÍSICAS) ---")
df_q5.select("dia_semana_pt", "media_venda_diaria").show(7, truncate=False)

print("\n✅ Camada Gold e Relatórios das Questões 4 e 5 concluídos com sucesso!")

Iniciando o processamento da Camada Gold...

--- QUESTÃO 4: TOP 10 CLIENTES FIÉIS (DIVERSIDADE >= 13) ---
+----+------------------------+-----------------+----------------------+------------------+----------+
|id  |legal_name              |ticket_medio     |diversidade_categorias|faturamento_total |frequencia|
+----+------------------------+-----------------+----------------------+------------------+----------+
|1527|Caio Farias             |46646.97666666667|13                    |699704.65         |15        |
|1581|Novaes Andrade S.A.     |44501.89749999999|14                    |534022.7699999999 |12        |
|1558|Natália Borges          |44348.15090909092|13                    |487829.6600000001 |11        |
|262 |Castro Cirino S.A.      |43821.1775       |14                    |525854.13         |12        |
|22  |Isadora Rios            |43180.09166666667|14                    |777241.65         |18        |
|1470|Bárbara Albuquerque     |42917.24952380952|14                   

In [0]:
# QUESTÃO 6: Previsão de Demanda para 'Bússola de Bordo 702'
import pandas as pd
from sklearn.metrics import mean_absolute_error

#tabelas
df_orders = spark.table("default.silver_orders").toPandas()
df_items = spark.table("default.silver_order_items").toPandas()
df_vars = spark.table("default.bronze_product_variants").toPandas()
df_prods = spark.table("default.bronze_products").toPandas()

#Filtrar o produto
bussola_id = df_prods[df_prods['name'].str.contains('Bússola de Bordo 702', case=False, na=False)]['id'].values[0]
vars_bussola = df_vars[df_vars['product_id'] == bussola_id]['id'].tolist()

df_vendas = df_orders.merge(df_items[df_items['product_variant_id'].isin(vars_bussola)], left_on='id', right_on='order_id')
df_vendas['placed_at'] = pd.to_datetime(df_vendas['placed_at'])

#Agrupamento mensal para preenche meses sem venda com 0
ts_mensal = df_vendas.set_index('placed_at').resample('MS')['quantity'].sum().fillna(0)

#Split treino/teste conforme premissas do enunciado
treino = ts_mensal[:'2025-12-31']
teste = ts_mensal['2026-01-01':'2026-03-31']

# 5. BASELINE: média móvel dos últimos 3 meses (sem usar dados futuros)
#    Para cada mês do Q1/2026, a previsão é a média dos 3 meses anteriores já conhecidos.
historico = treino.copy()
previsoes_baseline = []

for data_alvo in teste.index:
    media_3m = historico[-3:].mean()
    previsoes_baseline.append(media_3m)
    # "avança" o histórico incluindo o valor real do mês, só depois de já ter previsto
    historico = pd.concat([historico, pd.Series([teste[data_alvo]], index=[data_alvo])])

previsao_baseline = pd.Series(previsoes_baseline, index=teste.index)
soma_previsao_baseline = int(round(previsao_baseline.sum()))
mae_baseline = mean_absolute_error(teste, previsao_baseline)

print("--- QUESTÃO 6: BASELINE (MÉDIA MÓVEL 3 MESES) ---")
print(f"Previsão mensal (baseline): {previsao_baseline.round(2).to_dict()}")
print(f"Soma total arredondada Q1/2026 (baseline): {soma_previsao_baseline} unidades")
print(f"MAE do baseline: {mae_baseline:.2f}")

# 6. Comparação opcional: SARIMAX
from statsmodels.tsa.statespace.sarimax import SARIMAX

modelo = SARIMAX(treino, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12))
modelo_fit = modelo.fit(disp=False)
previsao_sarimax = modelo_fit.forecast(steps=3)
soma_previsao_sarimax = int(round(previsao_sarimax.sum()))
mae_sarimax = mean_absolute_error(teste, previsao_sarimax)

print("\n--- COMPARAÇÃO: SARIMAX ---")
print(f"Soma total arredondada Q1/2026 (SARIMAX): {soma_previsao_sarimax} unidades")
print(f"MAE do SARIMAX: {mae_sarimax:.2f}")

--- QUESTÃO 6: BASELINE (MÉDIA MÓVEL 3 MESES) ---
Previsão mensal (baseline): {Timestamp('2026-01-01 00:00:00'): 18.67, Timestamp('2026-02-01 00:00:00'): 30.33, Timestamp('2026-03-01 00:00:00'): 30.0}
Soma total arredondada Q1/2026 (baseline): 79 unidades
MAE do baseline: 14.22

--- COMPARAÇÃO: SARIMAX ---
Soma total arredondada Q1/2026 (SARIMAX): 95 unidades
MAE do SARIMAX: 19.21


In [0]:
#QUESTÃO 7: Sistema de Recomendação (Motor de Popa 1949)
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. Unir vendas com produtos (merge do pandas, não join do Spark)
df_vendas_all = df_orders.merge(df_items, left_on='id', right_on='order_id') \
                          .merge(df_vars, left_on='product_variant_id', right_on='id')

# 2. Matriz Usuário x Produto (1 comprou, 0 não comprou)
matriz_user_prod = pd.crosstab(df_vendas_all['customer_id'], df_vendas_all['product_id']).clip(upper=1)

# 3. Similaridade de Cosseno entre Produtos
matriz_sim = cosine_similarity(matriz_user_prod.T)
df_sim = pd.DataFrame(matriz_sim, index=matriz_user_prod.columns, columns=matriz_user_prod.columns)

# 4. Produto alvo: 'Motor de Popa 1949'
motor_id = df_prods[df_prods['name'].str.contains('Motor de Popa 1949', case=False, na=False)]['id'].values[0]

# 5. Ranking dos 5 produtos mais similares (excluindo o próprio motor)
ranking = df_sim[motor_id].drop(motor_id).sort_values(ascending=False).head(5)

print("--- QUESTÃO 7: TOP 5 PRODUTOS MAIS SIMILARES AO 'MOTOR DE POPA 1949' ---")
for pid, score in ranking.items():
    nome = df_prods[df_prods['id'] == pid]['name'].values[0]
    print(f"{nome} — Score Cosseno: {score:.4f}")

--- QUESTÃO 7: TOP 5 PRODUTOS MAIS SIMILARES AO 'MOTOR DE POPA 1949' ---
Vela Mestra 1913 — Score Cosseno: 0.2046
Cabo Náutico 2105 — Score Cosseno: 0.1898
Motor de Popa 6014 — Score Cosseno: 0.1872
asdf — Score Cosseno: 0.1854
Âncora Bruce 7665 — Score Cosseno: 0.1853
